# 05. XGBoost Regression

**Objectif** : Prédire les prix immobiliers avec XGBoost optimisé

**Pour les débutants** : Ce notebook explique XGBoost simplement avec optimisation avancée

In [1]:
# Importer les bibliothèques
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Modélisation XGBoost
import xgboost as xgb
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler
from scipy import stats
import time

%matplotlib inline

# Configuration du style
try:
    plt.style.use('seaborn-v0_8')
except:
    try:
        plt.style.use('seaborn')
    except:
        plt.style.use('default')

print("📚 Bibliothèques importées avec succès !")
print("🚀 XGBoost Regression prêt")
print(f"📦 Version XGBoost: {xgb.__version__}")

📚 Bibliothèques importées avec succès !
🚀 XGBoost Regression prêt
📦 Version XGBoost: 3.2.0


In [2]:
# Charger et préparer les données
df = pd.read_csv("../data/real_estate_processed.csv")
print(f"Données originales: {df.shape}")

# Nettoyage
df_clean = df[(df['price'] >= 50) & (df['price'] <= 10000000)].copy()
print(f"Données après nettoyage: {df_clean.shape}")
print(f"Supprimées: {len(df) - len(df_clean)} annonces")

df = df_clean

print(f"\nStatistiques des prix:")
print(f"Prix moyen: {df['price'].mean():.0f} DT")
print(f"Prix médian: {df['price'].median():.0f} DT")
print(f"Prix min: {df['price'].min():.0f} DT")
print(f"Prix max: {df['price'].max():.0f} DT")
print(f"Écart-type: {df['price'].std():.0f} DT")

# Analyse de la distribution
print(f"\n📊 Distribution des prix:")
print(f"   Skewness: {stats.skew(df['price']):.4f}")
print(f"   Kurtosis: {stats.kurtosis(df['price']):.4f}")

df.head(3)

Données originales: (5653, 12)
Données après nettoyage: (5601, 12)
Supprimées: 52 annonces

Statistiques des prix:
Prix moyen: 176410 DT
Prix médian: 3500 DT
Prix min: 55 DT
Prix max: 6300000 DT
Écart-type: 300082 DT

📊 Distribution des prix:
   Skewness: 4.8640
   Kurtosis: 58.2681


,title,price_text,price,category,city,location,type_transaction,rooms,post_time,post_date,post_month,post_year
0,À louer – Bureaux neufs S+1 et S+2 à Monastir ...,650 DT,650,0,Monastir,Monastir,0,1,2/4/26 12:37,02-04-26,2,2026
1,S+1 haut standing pour la saison universitaire,850 DT,850,0,Monastir,Monastir,0,1,8/30/25 10:49,08-30-25,8,2025
2,à vendre s+3 haut standing directement au prom...,350000 DT,350000,1,Monastir,Bekalta,1,3,7/30/25 12:45,07-30-25,7,2025


In [3]:
# Feature Engineering optimisé pour XGBoost
print("=== FEATURE ENGINEERING POUR XGBOOST ===")

# Features de base
numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()
X_base = df[numeric_columns].drop('price', axis=1)
y = df['price']

print(f"Colonnes disponibles: {X_base.columns.tolist()}")

# Création de features optimisées pour XGBoost
X = X_base.copy()

# Features temporelles avancées (créer quarter avant de l'utiliser)
X['quarter'] = ((X['post_month'] - 1) // 3) + 1
X['is_summer'] = (X['post_month'].isin([6, 7, 8])).astype(int)
X['is_winter'] = (X['post_month'].isin([12, 1, 2])).astype(int)
X['is_spring'] = (X['post_month'].isin([3, 4, 5])).astype(int)
X['is_autumn'] = (X['post_month'].isin([9, 10, 11])).astype(int)
X['is_weekend'] = (X['post_month'] % 7 >= 5).astype(int)
X['is_quarter_end'] = (X['post_month'].isin([3, 6, 9, 12])).astype(int)

# Features d'interaction complexes (XGBoost gère très bien les interactions)
X['category_transaction_interaction'] = X['category'] * X['type_transaction']
X['category_month_interaction'] = X['category'] * X['post_month']
X['transaction_month_interaction'] = X['type_transaction'] * X['post_month']
X['category_year_interaction'] = X['category'] * X['post_year']
X['transaction_year_interaction'] = X['type_transaction'] * X['post_year']
X['month_year_interaction'] = X['post_month'] * X['post_year']

# Features polynomiales (XGBoost peut bénéficier de transformations polynomiales)
X['category_squared'] = X['category'] ** 2
X['type_transaction_squared'] = X['type_transaction'] ** 2
X['month_squared'] = X['post_month'] ** 2
X['year_squared'] = X['post_year'] ** 2
X['category_cubed'] = X['category'] ** 3
X['type_transaction_cubed'] = X['type_transaction'] ** 3

# Features de ratio et division
X['category_per_month'] = X['category'] / (X['post_month'] + 1)
X['transaction_per_year'] = X['type_transaction'] / (X['post_year'] - 2024 + 1)
X['category_per_transaction'] = X['category'] / (X['type_transaction'] + 1)
X['month_per_year'] = X['post_month'] / 12

# Target encoding multi-niveaux (très puissant pour XGBoost)
price_by_category = df.groupby('category')['price'].agg(['mean', 'median', 'std'])
price_by_transaction = df.groupby('type_transaction')['price'].agg(['mean', 'median', 'std'])
price_by_month = df.groupby('post_month')['price'].agg(['mean', 'median', 'std'])
price_by_year = df.groupby('post_year')['price'].agg(['mean', 'median', 'std'])

# Ajouter la colonne quarter au dataframe original pour le groupby
df_with_quarter = df.copy()
df_with_quarter['quarter'] = ((df_with_quarter['post_month'] - 1) // 3) + 1
price_by_quarter = df_with_quarter.groupby('quarter')['price'].agg(['mean', 'median', 'std'])

# Ajouter les target encodings
X['category_price_mean'] = X['category'].map(price_by_category['mean'])
X['category_price_median'] = X['category'].map(price_by_category['median'])
X['category_price_std'] = X['category'].map(price_by_category['std'])
X['transaction_price_mean'] = X['type_transaction'].map(price_by_transaction['mean'])
X['transaction_price_median'] = X['type_transaction'].map(price_by_transaction['median'])
X['transaction_price_std'] = X['type_transaction'].map(price_by_transaction['std'])
X['month_price_mean'] = X['post_month'].map(price_by_month['mean'])
X['month_price_median'] = X['post_month'].map(price_by_month['median'])
X['month_price_std'] = X['post_month'].map(price_by_month['std'])
X['year_price_mean'] = X['post_year'].map(price_by_year['mean'])
X['year_price_median'] = X['post_year'].map(price_by_year['median'])
X['year_price_std'] = X['post_year'].map(price_by_year['std'])
X['quarter_price_mean'] = X['quarter'].map(price_by_quarter['mean'])
X['quarter_price_median'] = X['quarter'].map(price_by_quarter['median'])
X['quarter_price_std'] = X['quarter'].map(price_by_quarter['std'])

# Features logarithmiques (uniquement pour les colonnes qui existent)
X['log_category'] = np.log1p(X['category'])
X['log_month'] = np.log1p(X['post_month'])
X['log_year_diff'] = np.log1p(X['post_year'] - 2024)

# Ajouter log_rooms et log_location seulement si ces colonnes existent
if 'rooms' in X.columns:
    X['log_rooms'] = np.log1p(X['rooms'])
if 'location' in X.columns:
    X['log_location'] = np.log1p(X['location'])

# Features de différence et de changement
X['category_minus_transaction'] = X['category'] - X['type_transaction']
X['month_minus_year'] = X['post_month'] - (X['post_year'] - 2024)
X['price_category_ratio'] = X['category_price_mean'] / (X['transaction_price_mean'] + 1)

# Remplir les valeurs NaN avec 0
X = X.fillna(0)

print(f"Features créées: {X_base.shape[1]} -> {X.shape[1]} (+{X.shape[1] - X_base.shape[1]} nouvelles)")

# Transformation de la target
y_log = np.log1p(y)

# Division des données
X_train, X_test, y_train, y_test = train_test_split(X, y_log, test_size=0.2, random_state=42)

print(f"Données divisées: {X_train.shape[0]} entraînement, {X_test.shape[0]} test")
print(f"Features finales: {X.shape[1]}")

=== FEATURE ENGINEERING POUR XGBOOST ===
Colonnes disponibles: ['category', 'type_transaction', 'post_month', 'post_year']
Features créées: 4 -> 48 (+44 nouvelles)
Données divisées: 4480 entraînement, 1121 test
Features finales: 48


## XGBoost Expliqué

### Qu'est-ce que XGBoost ?

**XGBoost = eXtreme Gradient Boosting**

C'est une implémentation optimisée de Gradient Boosting qui est : 
- **10x plus rapide** que les autres implémentations
- **Plus précise** grâce à des optimisations algorithmiques
- **Parallélisable** pour utiliser plusieurs CPU

### Comment ça marche ?
1. **Arbres faibles** : Commence avec des arbres très simples
2. **Gradient** : Calcule le gradient de la perte
3. **Boosting** : Ajoute des arbres pour corriger les erreurs
4. **Learning rate** : Contrôle la contribution de chaque arbre
5. **Regularisation** : Évite l'overfitting (L1, L2)

### Avantages XGBoost :
- ✅ **Performance** : Souvent le meilleur sur données tabulaires
- ✅ **Vitesse** : Optimisé et parallélisable
- ✅ **Flexibilité** : Fonctions de perte personnalisées
- ✅ **Robustesse** : Gère les valeurs manquantes
- ✅ **Interprétabilité** : Feature importance intégrée

### Hyperparamètres clés :
- `n_estimators` : Nombre d'arbres
- `max_depth` : Profondeur maximale des arbres
- `learning_rate` : Taux d'apprentissage
- `subsample` : Fraction d'échantillons
- `colsample_bytree` : Fraction de features
- `reg_alpha` : Regularisation L1
- `reg_lambda` : Regularisation L2

In [4]:
# XGBoost - Version simple
print("=== XGBOOST - VERSION SIMPLE ===")

# Créer et entraîner le modèle
start_time = time.time()

xgb_simple = xgb.XGBRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    objective='reg:squarederror'
)

xgb_simple.fit(X_train, y_train)
training_time = time.time() - start_time

print(f"⏱️ Temps d'entraînement: {training_time:.2f} secondes")
print(f"🌳 Nombre d'arbres: {xgb_simple.n_estimators}")
print(f"📏 Profondeur max: {xgb_simple.max_depth}")
print(f"📚 Learning rate: {xgb_simple.learning_rate}")
print(f"✅ Modèle entraîné !")

=== XGBOOST - VERSION SIMPLE ===
⏱️ Temps d'entraînement: 0.36 secondes
🌳 Nombre d'arbres: 200
📏 Profondeur max: 6
📚 Learning rate: 0.1
✅ Modèle entraîné !


In [5]:
# Évaluation du modèle simple
print("=== ÉVALUATION MODÈLE SIMPLE ===")

# Prédictions
y_train_pred = xgb_simple.predict(X_train)
y_test_pred = xgb_simple.predict(X_test)

# Conversion à l'échelle originale
y_train_orig = np.expm1(y_train)
y_test_orig = np.expm1(y_test)
y_train_pred_orig = np.expm1(y_train_pred)
y_test_pred_orig = np.expm1(y_test_pred)

# Métriques
train_r2 = r2_score(y_train_orig, y_train_pred_orig)
test_r2 = r2_score(y_test_orig, y_test_pred_orig)
train_rmse = np.sqrt(mean_squared_error(y_train_orig, y_train_pred_orig))
test_rmse = np.sqrt(mean_squared_error(y_test_orig, y_test_pred_orig))
train_mae = mean_absolute_error(y_train_orig, y_train_pred_orig)
test_mae = mean_absolute_error(y_test_orig, y_test_pred_orig)

print(f"📊 PERFORMANCE XGBOOST:")
print(f"   R² Train: {train_r2:.4f}")
print(f"   R² Test:  {test_r2:.4f}")
print(f"   RMSE Train: {train_rmse:,.0f} DT")
print(f"   RMSE Test:  {test_rmse:,.0f} DT")
print(f"   MAE Train:  {train_mae:,.0f} DT")
print(f"   MAE Test:   {test_mae:,.0f} DT")

# Analyse de l'overfitting
overfitting_r2 = train_r2 - test_r2
print(f"\n🔍 ANALYSE OVERFITTING:")
print(f"   Différence R² (Train-Test): {overfitting_r2:.4f}")

if overfitting_r2 > 0.1:
    print(f"   ⚠️ Overfitting détecté !")
elif overfitting_r2 > 0.05:
    print(f"   🟡 Léger overfitting")
else:
    print(f"   ✅ Bon équilibre")

# Qualité du modèle
if test_r2 > 0.7:
    quality = "🌟 Excellente"
elif test_r2 > 0.5:
    quality = "✅ Bonne"
elif test_r2 > 0.3:
    quality = "⚠️ Moyenne"
else:
    quality = "📉 Faible"

print(f"\n🏆 QUALITÉ: {quality}")

=== ÉVALUATION MODÈLE SIMPLE ===
📊 PERFORMANCE XGBOOST:
   R² Train: 0.4224
   R² Test:  0.4608
   RMSE Train: 233,085 DT
   RMSE Test:  199,281 DT
   MAE Train:  86,571 DT
   MAE Test:   80,882 DT

🔍 ANALYSE OVERFITTING:
   Différence R² (Train-Test): -0.0384
   ✅ Bon équilibre

🏆 QUALITÉ: ⚠️ Moyenne


In [6]:
# Sauvegarde du modèle XGBoost
print("=== SAUVEGARDE DU MODÈLE XGBOOST ===")

import joblib
import os

# Créer le dossier models s'il n'existe pas
if not os.path.exists('../models'):
    os.makedirs('../models')

# Sauvegarder le meilleur modèle
joblib.dump(xgb_simple, '../models/xgboost_best_model.pkl')

# Sauvegarder les résultats et métadonnées
xgb_results = {
    'model_name': 'XGBoost',
    'version': 'Simple',
    'r2_train': train_r2,
    'r2_test': test_r2,
    'rmse_train': train_rmse,
    'rmse_test': test_rmse,
    'mae_train': train_mae,
    'mae_test': test_mae,
    'training_time': training_time,
    'n_features': X.shape[1],
    'feature_columns': list(X.columns),
    'xgb_version': xgb.__version__,
    'quality': quality
}

joblib.dump(xgb_results, '../models/xgboost_results.pkl')

print(f"   ✅ Modèle sauvegardé: ../models/xgboost_best_model.pkl")
print(f"   ✅ Résultats sauvegardés: ../models/xgboost_results.pkl")
print(f"\n🎉 XGBoost Regression terminé avec succès !")

=== SAUVEGARDE DU MODÈLE XGBOOST ===
   ✅ Modèle sauvegardé: ../models/xgboost_best_model.pkl
   ✅ Résultats sauvegardés: ../models/xgboost_results.pkl

🎉 XGBoost Regression terminé avec succès !
